# DJF ITCZ wind-divergence diagnostic

This notebook builds a DJF ITCZ diagnostic using 850 hPa u/v winds and the Ramotubei wind-divergence approach.

- DJF labeling follows Dec(year-1) + Jan(year) + Feb(year) = DJF(year).
- The ITCZ line is the latitude of minimum divergence within 20S-20N for each longitude.
- No precipitation is used in the extraction.


In [4]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import metpy.calc as mpcalc
from metpy.units import units

MPLCONFIGDIR = Path('/tmp/matplotlib')
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPLCONFIGDIR))

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']

FIGURE_DIR = Path('.')
U_V_PATH = Path('/Users/rizzie/TugasAkhir/data_processing/external/ClimateData/era5-monthly/u_v_global850.nc')
NINO34_PATH = Path('/Users/rizzie/TugasAkhir/data_processing/external/ClimateData/index-monthly/nino34.anom.csv')
NINO34_COLUMN = '   Nino Anom 3.4 Index  using ersstv5 from CPC  missing value -99.99 https://psl.noaa.gov/data/timeseries/month/'

ALL_YEARS = np.arange(1981, 2021)
P1_YEARS = np.arange(1981, 2007)
P2_YEARS = np.arange(2007, 2021)

PERIOD_ORDER = ['All', 'P1', 'P2']
PHASE_ORDER = ['Climatology', 'El Niño', 'La Niña', 'Neutral']

PHASE_COLORS = {
    'Climatology': 'black',
    'El Niño': 'red',
    'La Niña': 'blue',
    'Neutral': 'green',
}
PERIOD_STYLES = {
    'All': '-',
    'P1': '--',
    'P2': ':',
}
PHASE_LABELS = {
    'Climatology': 'Climatology',
    'El Niño': 'El Niño',
    'La Niña': 'La Niña',
    'Neutral': 'Neutral',
}
PERIOD_LABELS = {
    'All': 'All',
    'P1': 'P1 (1981-2006)',
    'P2': 'P2 (2007-2020)',
}

GLOBAL_EXTENT = [0, 360, -25, 25]
GLOBAL_LON_TICKS = np.arange(0, 360, 30)
GLOBAL_LAT_TICKS = np.arange(-25, 26, 5)
MC_EXTENT = [90, 155, -20, 20]
MC_LON_TICKS = np.arange(90, 156, 10)
MC_LAT_TICKS = np.arange(-20, 21, 5)


In [5]:
ds

<xarray.Dataset> Size: 1GB
Dimensions:         (valid_time: 123, pressure_level: 1, latitude: 721,
                     longitude: 1440)
Coordinates:
    number          int64 8B ...
  * valid_time      (valid_time) datetime64[ns] 984B 1980-01-01 ... 2020-12-01
  * pressure_level  (pressure_level) float64 8B 850.0
  * latitude        (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude       (longitude) float64 12kB 0.0 0.25 0.5 ... 359.2 359.5 359.8
    expver          (valid_time) <U4 2kB dask.array<chunksize=(12,), meta=np.ndarray>
Data variables:
    u               (valid_time, pressure_level, latitude, longitude) float32 511MB dask.array<chunksize=(12, 1, 241, 480), meta=np.ndarray>
    v               (valid_time, pressure_level, latitude, longitude) float32 511MB dask.array<chunksize=(12, 1, 241, 480), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-04-13T21:44 GRIB to CDM+CF via cfgrib-0.9.1...

In [6]:
# Load the ERA5 850 hPa wind file and the Niño3.4 index series.
# The ERA5 file already contains DJF months only, but the season labeling below is still explicit.
ds = xr.open_dataset(U_V_PATH)[['u', 'v']]
if 'pressure_level' in ds.dims:
    if ds.sizes['pressure_level'] == 1:
        ds = ds.isel(pressure_level=0, drop=True)
    else:
        ds = ds.sel(pressure_level=850.0)
        if 'pressure_level' in ds.dims:
            ds = ds.isel(pressure_level=0, drop=True)
ds = ds.rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
ds = ds.assign_coords(lon=(ds.lon % 360)).sortby('lon').sortby('lat')
ds = ds.sel(time=slice('1980-12-01', '2020-02-29'))

u850 = ds['u']
v850 = ds['v']

nino34 = pd.read_csv(
    NINO34_PATH,
    usecols=['Date', NINO34_COLUMN],
    parse_dates=['Date'],
)
nino34 = nino34.set_index('Date').loc['1980-12-01':'2020-02-29'].copy()
nino34 = nino34[nino34.index.month.isin([12, 1, 2])].copy()
nino34[NINO34_COLUMN] = pd.to_numeric(nino34[NINO34_COLUMN], errors='coerce').replace([-9999.0, -99.99], np.nan)

print('u850 dimensions:', dict(u850.sizes))
print('v850 dimensions:', dict(v850.sizes))
print('Nino3.4 DJF rows:', len(nino34))


u850 dimensions: {'time': 120, 'lat': 721, 'lon': 1440}
v850 dimensions: {'time': 120, 'lat': 721, 'lon': 1440}
Nino3.4 DJF rows: 120


In [7]:
def make_djf_means(da: xr.DataArray, start_year: int = 1981, end_year: int = 2020) -> xr.DataArray:
    djf = da.sel(time=da.time.dt.month.isin([12, 1, 2])).copy()
    time_index = pd.DatetimeIndex(djf['time'].values)
    djf_year_values = np.where(time_index.month == 12, time_index.year + 1, time_index.year).astype(int)
    djf = djf.assign_coords(djf_year=('time', djf_year_values))
    season_counts = pd.Series(djf_year_values).value_counts().sort_index()
    complete_years = season_counts[season_counts == 3].index.to_numpy(dtype=int)
    complete_years = complete_years[(complete_years >= start_year) & (complete_years <= end_year)]
    if complete_years.size == 0:
        raise ValueError(f'No complete DJF seasons found for {da.name or "field"}')
    return djf.groupby('djf_year').mean('time').sel(djf_year=complete_years)

def make_djf_nino34_mean(df: pd.DataFrame, column: str, start_year: int = 1981, end_year: int = 2020) -> pd.Series:
    subset = df.loc[f'{start_year - 1}-12-01':f'{end_year}-02-29', [column]].copy()
    subset = subset[subset.index.month.isin([12, 1, 2])].copy()
    subset['djf_year'] = subset.index.year + (subset.index.month == 12).astype('int8')
    valid_counts = subset.groupby('djf_year')[column].apply(lambda s: s.notna().sum())
    complete_years = valid_counts[valid_counts == 3].index.to_numpy(dtype=int)
    complete_years = complete_years[(complete_years >= start_year) & (complete_years <= end_year)]
    if complete_years.size == 0:
        raise ValueError('No complete DJF seasons found for Niño3.4')
    return subset.groupby('djf_year')[column].mean().loc[complete_years]

def classify_enso(value: float) -> str:
    if pd.isna(value):
        return np.nan
    if value >= 0.5:
        return 'El Niño'
    if value <= -0.5:
        return 'La Niña'
    return 'Neutral'

def build_event_table(nino_series: pd.Series, years, period_label: str) -> pd.DataFrame:
    years = np.asarray(years, dtype=int)
    table = pd.DataFrame({'djf_year': years, 'nino34_djf_mean': nino_series.reindex(years).to_numpy()})
    table['enso_phase'] = table['nino34_djf_mean'].map(classify_enso)
    table['period'] = period_label
    return table

def compute_divergence(u_field: xr.DataArray, v_field: xr.DataArray) -> xr.DataArray:
    lons = np.asarray(u_field['lon'].values, dtype=float)
    lats = np.asarray(u_field['lat'].values, dtype=float)
    dx, dy = mpcalc.lat_lon_grid_deltas(lons, lats)
    div = mpcalc.divergence(
        u_field.values * units('m/s'),
        v_field.values * units('m/s'),
        dx=dx,
        dy=dy,
    )
    return xr.DataArray(
        div.magnitude,
        coords={'lat': u_field['lat'], 'lon': u_field['lon']},
        dims=('lat', 'lon'),
        name='divergence',
        attrs={'units': str(div.units)},
    )

# Note: wind-divergence ITCZ is generally more reliable over oceans than continents.
def extract_min_div_line(div_field: xr.DataArray, lat_min: float = -20.0, lat_max: float = 20.0) -> xr.DataArray:
    band = div_field.sel(lat=slice(lat_min, lat_max))
    line = band.idxmin('lat')
    line.name = 'itcz_latitude'
    return line

def composite_wind_and_divergence(u_djf: xr.DataArray, v_djf: xr.DataArray, years) -> tuple[xr.DataArray, xr.DataArray, xr.DataArray]:
    years = np.asarray(years, dtype=int)
    u_comp = u_djf.sel(djf_year=years).mean('djf_year').transpose('lat', 'lon')
    v_comp = v_djf.sel(djf_year=years).mean('djf_year').transpose('lat', 'lon')
    div_comp = compute_divergence(u_comp, v_comp)
    return u_comp, v_comp, div_comp


## DJF means and ENSO classification

The DJF seasonal means are built after year labeling, so Dec 1980 + Jan 1981 + Feb 1981 becomes DJF1981.


In [8]:
u_djf = make_djf_means(u850).transpose('djf_year', 'lat', 'lon')
v_djf = make_djf_means(v850).transpose('djf_year', 'lat', 'lon')
nino34_djf = make_djf_nino34_mean(nino34, NINO34_COLUMN)

common_years = np.intersect1d(u_djf['djf_year'].values.astype(int), nino34_djf.index.to_numpy(dtype=int))
common_years = common_years[(common_years >= 1981) & (common_years <= 2020)]
u_djf = u_djf.sel(djf_year=common_years)
v_djf = v_djf.sel(djf_year=common_years)
nino34_djf = nino34_djf.reindex(common_years)

events_long = pd.concat(
    [
        build_event_table(nino34_djf, ALL_YEARS, 'All'),
        build_event_table(nino34_djf, P1_YEARS, 'P1'),
        build_event_table(nino34_djf, P2_YEARS, 'P2'),
    ],
    ignore_index=True,
)
events_long['period'] = pd.Categorical(events_long['period'], categories=PERIOD_ORDER, ordered=True)
events_long = events_long.sort_values(['period', 'djf_year']).reset_index(drop=True)

counts = (
    events_long.groupby(['period', 'enso_phase'])
    .size()
    .unstack(fill_value=0)
    .reindex(index=PERIOD_ORDER, columns=['El Niño', 'La Niña', 'Neutral'])
)

print('DJF ENSO year table:')
print(events_long.to_string(index=False))
print('\nDJF ENSO counts:')
print(counts.to_string())

period_years = {'All': ALL_YEARS, 'P1': P1_YEARS, 'P2': P2_YEARS}
phase_years_map = {}
for period in PERIOD_ORDER:
    period_table = events_long[events_long['period'] == period]
    phase_years_map[period] = {
        'Climatology': period_years[period],
        'El Niño': period_table.loc[period_table['enso_phase'] == 'El Niño', 'djf_year'].to_numpy(),
        'La Niña': period_table.loc[period_table['enso_phase'] == 'La Niña', 'djf_year'].to_numpy(),
        'Neutral': period_table.loc[period_table['enso_phase'] == 'Neutral', 'djf_year'].to_numpy(),
    }


KeyboardInterrupt: 

## DJF composites and ITCZ extraction

Each composite is formed from seasonal-mean 850 hPa u/v winds first, then divergence is computed from the seasonal mean using MetPy. The ITCZ line is the latitude of minimum divergence within 20S-20N for each longitude.


In [ ]:
u_composites = {}
v_composites = {}
div_composites = {}
itcz_lines = {}
itcz_zonal = {}

for period in PERIOD_ORDER:
    u_composites[period] = {}
    v_composites[period] = {}
    div_composites[period] = {}
    itcz_lines[period] = {}
    itcz_zonal[period] = {}
    for phase in PHASE_ORDER:
        u_comp, v_comp, div_comp = composite_wind_and_divergence(u_djf, v_djf, phase_years_map[period][phase])
        line = extract_min_div_line(div_comp, lat_min=-20.0, lat_max=20.0)
        u_composites[period][phase] = u_comp
        v_composites[period][phase] = v_comp
        div_composites[period][phase] = div_comp
        itcz_lines[period][phase] = line
        itcz_zonal[period][phase] = float(line.mean('lon', skipna=True))

summary_rows = []
for period in PERIOD_ORDER:
    for phase in PHASE_ORDER:
        summary_rows.append({
            'period': period,
            'phase': phase,
            'n_years': int(len(phase_years_map[period][phase])),
            'zonal_itcz_lat_deg': itcz_zonal[period][phase],
        })

itcz_summary = pd.DataFrame(summary_rows)
print('Zonal ITCZ positions (mean latitude over longitude):')
print(itcz_summary.to_string(index=False))


## Maps

Color encodes ENSO phase and line style encodes period. The figure uses Cartopy coastlines at 10 m resolution and explicit lon/lat ticks.


In [ ]:
def plot_itcz_map(itcz_lines_by_period, title: str, extent, xticks, yticks, save_path: Path, figsize=(13.5, 6.5)) -> None:
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    for period in PERIOD_ORDER:
        for phase in PHASE_ORDER:
            line = itcz_lines_by_period[period][phase]
            ax.plot(
                line['lon'].values,
                line.values,
                color=PHASE_COLORS[phase],
                linestyle=PERIOD_STYLES[period],
                linewidth=2.0,
                transform=ccrs.PlateCarree(),
                zorder=3,
            )

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=2)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=2)
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)

    phase_handles = [Line2D([0], [0], color=PHASE_COLORS[phase], lw=2.2, label=PHASE_LABELS[phase]) for phase in PHASE_ORDER]
    period_handles = [Line2D([0], [0], color='black', lw=2.2, linestyle=PERIOD_STYLES[period], label=PERIOD_LABELS[period]) for period in PERIOD_ORDER]
    phase_legend = ax.legend(handles=phase_handles, loc='lower left', frameon=True, fontsize=9, title='Phase')
    ax.add_artist(phase_legend)
    ax.legend(handles=period_handles, loc='lower right', frameon=True, fontsize=9, title='Period')

    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved figure: {save_path}')


In [ ]:
plot_itcz_map(
    itcz_lines,
    title='DJF 850 hPa wind-divergence ITCZ - Global',
    extent=GLOBAL_EXTENT,
    xticks=GLOBAL_LON_TICKS,
    yticks=GLOBAL_LAT_TICKS,
    save_path=FIGURE_DIR / 'djf_itcz_global_mindiv_all_p1_p2.png',
    figsize=(13.5, 6.5),
)


In [ ]:
plot_itcz_map(
    itcz_lines,
    title='DJF 850 hPa wind-divergence ITCZ - Maritime Continent',
    extent=MC_EXTENT,
    xticks=MC_LON_TICKS,
    yticks=MC_LAT_TICKS,
    save_path=FIGURE_DIR / 'djf_itcz_mc_90E_155E_mindiv_all_p1_p2.png',
    figsize=(12.5, 6.5),
)
